In [1]:
!pip install imdlib netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 25.4 MB/s eta 0:00:00


In [2]:
import pandas as pd, numpy as np, requests, time

districts = pd.DataFrame([
    ("ratnagiri", "Ratnagiri", 16.99, 73.30, "coastal"),
    ("nagpur",    "Nagpur",    21.15, 79.09, "interior"),
    ("kodagu",    "Kodagu",    12.42, 75.74, "ghats"),
], columns=["id", "name", "lat", "lon", "loc_class"])
districts

,id,name,lat,lon,loc_class
0,ratnagiri,Ratnagiri,16.99,73.30,coastal
1,nagpur,Nagpur,21.15,79.09,interior
2,kodagu,Kodagu,12.42,75.74,ghats


In [3]:
def fetch_forecast(lat, lon, start, end):
    r = requests.get(
        "https://historical-forecast-api.open-meteo.com/v1/forecast",
        params=dict(latitude=lat, longitude=lon, start_date=start,
                    end_date=end, hourly="precipitation", timezone="UTC"),
        timeout=60)
    r.raise_for_status()
    h = r.json()["hourly"]
    s = pd.Series(h["precipitation"], index=pd.to_datetime(h["time"]))
    d = s.resample("24h", offset="3h").sum()      # 03 UTC to 03 UTC
    d.index = d.index.normalize() + pd.Timedelta(days=1)
    return d

row = districts.iloc[0]
fc = fetch_forecast(row.lat, row.lon, "2024-07-01", "2024-07-31")
fc.head(10)

,0
2024-07-01,2.0
2024-07-02,15.6
2024-07-03,30.1
2024-07-04,14.7
2024-07-05,16.4
2024-07-06,30.4
2024-07-07,62.5
2024-07-08,31.9
2024-07-09,46.7
2024-07-10,29.6


In [4]:
import imdlib as imd

def has_forecast(year):
    try:
        s = fetch_forecast(16.99, 73.30, f"{year}-07-10", f"{year}-07-14")
        return s.notna().sum() > 0 and s.sum() > 0
    except Exception as e:
        return False

def has_imd(year):
    try:
        imd.get_data("rain", year, year, fn_format="yearwise")
        d = imd.open_data("rain", year, year, "yearwise").get_xarray()
        v = d["rain"].sel(lat=16.99, lon=73.30, method="nearest")
        v = v.sel(time=slice(f"{year}-07-10", f"{year}-07-14")).values
        return bool((v > -900).any())
    except Exception as e:
        return False

for y in range(2019, 2027):
    print(y, "forecast:", has_forecast(y), "| IMD:", has_imd(y))
    time.sleep(1)

Downloading: rain for year 2019
Download Successful !!!
2019 forecast: True | IMD: True
Downloading: rain for year 2020
Download Successful !!!
2020 forecast: True | IMD: True
Downloading: rain for year 2021
Download Successful !!!
2021 forecast: True | IMD: True
Downloading: rain for year 2022
Download Successful !!!
2022 forecast: True | IMD: True
Downloading: rain for year 2023
Download Successful !!!
2023 forecast: True | IMD: True
Downloading: rain for year 2024
Download Successful !!!
2024 forecast: True | IMD: True
Downloading: rain for year 2025
Download Successful !!!
2025 forecast: True | IMD: True
Downloading: rain for year 2026
2026 forecast: True | IMD: False


In [5]:
from io import StringIO
csv_text = """id,name,state,lat,lon,loc_class,core
kodagu,Kodagu,Karnataka,12.42,75.74,ghats,0
idukki,Idukki,Kerala,9.85,76.97,ghats,0
nilgiris,Nilgiris,Tamil Nadu,11.41,76.70,ghats,0
pune,Pune,Maharashtra,18.52,73.86,ghats,0
satara,Satara,Maharashtra,17.68,74.00,ghats,0
mumbai,Mumbai,Maharashtra,19.08,72.88,coastal,0
ratnagiri,Ratnagiri,Maharashtra,16.99,73.30,coastal,0
north_goa,North Goa,Goa,15.50,73.83,coastal,0
dakshina_kannada,Dakshina Kannada,Karnataka,12.91,74.86,coastal,0
kozhikode,Kozhikode,Kerala,11.26,75.78,coastal,0
thiruvananthapuram,Thiruvananthapuram,Kerala,8.52,76.94,coastal,0
puri,Puri,Odisha,19.81,85.83,coastal,0
kolkata,Kolkata,West Bengal,22.57,88.36,coastal,0
nagpur,Nagpur,Maharashtra,21.15,79.09,interior,1
bhopal,Bhopal,Madhya Pradesh,23.26,77.41,interior,1
raipur,Raipur,Chhattisgarh,21.25,81.63,interior,1
ahmedabad,Ahmedabad,Gujarat,23.02,72.57,interior,1
patna,Patna,Bihar,25.59,85.14,interior,1
hyderabad,Hyderabad,Telangana,17.39,78.49,interior,0
bengaluru,Bengaluru,Karnataka,12.97,77.59,interior,0
jaipur,Jaipur,Rajasthan,26.91,75.79,interior,0
lucknow,Lucknow,Uttar Pradesh,26.85,80.95,interior,0
ranchi,Ranchi,Jharkhand,23.34,85.31,interior,0
dehradun,Dehradun,Uttarakhand,30.32,78.03,north,0
shimla,Shimla,Himachal Pradesh,31.10,77.17,north,0
east_khasi_hills,East Khasi Hills,Meghalaya,25.27,91.73,north,0
"""
districts = pd.read_csv(StringIO(csv_text))
print(len(districts), "districts")

26 districts


In [6]:
YEARS = list(range(2019, 2026))
ds = imd.open_data("rain", 2019, 2025, "yearwise").get_xarray()

rows = []
for r in districts.itertuples():
    s = ds["rain"].sel(lat=r.lat, lon=r.lon, method="nearest").to_series()
    s = s.where(s > -900)
    s = s[(s.index.month >= 6) & (s.index.month <= 9)]
    rows.append(pd.DataFrame({"date": pd.to_datetime(s.index), "district": r.id, "obs": s.values}))
obs_all = pd.concat(rows)
print(obs_all.shape)
print(obs_all.groupby("district").obs.count().head())

(22204, 3)
district
ahmedabad           854
bengaluru           854
bhopal              854
dakshina_kannada    854
dehradun            854
Name: obs, dtype: int64


In [7]:
def get_fc(r, y):
    for attempt in range(3):
        try:
            s = fetch_forecast(r.lat, r.lon, f"{y}-05-31", f"{y}-10-01")
            return s[(s.index >= f"{y}-06-01") & (s.index <= f"{y}-09-30")]
        except Exception as e:
            time.sleep(5)
    return None

parts = []
for r in districts.itertuples():
    for y in YEARS:
        s = get_fc(r, y)
        if s is None:
            print("FAILED", r.id, y)
            continue
        parts.append(pd.DataFrame({"date": s.index, "district": r.id, "fc": s.values}))
        time.sleep(0.5)
    print("done", r.id)

fc_all = pd.concat(parts)
fc_all["date"] = pd.to_datetime(fc_all["date"])
print(fc_all.shape)

done kodagu
done idukki
done nilgiris
done pune
done satara
done mumbai
done ratnagiri
done north_goa
done dakshina_kannada
done kozhikode
done thiruvananthapuram
done puri
done kolkata
done nagpur
done bhopal
done raipur
done ahmedabad
done patna
done hyderabad
done bengaluru
done jaipur
done lucknow
done ranchi
done dehradun
done shimla
done east_khasi_hills
(22204, 3)


In [8]:
df = fc_all.merge(obs_all, on=["date", "district"]).dropna()
df = df.merge(districts[["id", "loc_class", "core"]], left_on="district", right_on="id").drop(columns="id")
print(df.shape)
print(df.head())

# which day-shift lines the forecast up best with the observations?
for k in (-1, 0, 1):
    tmp = fc_all.copy()
    tmp["date"] = tmp["date"] + pd.Timedelta(days=k)
    m = tmp.merge(obs_all, on=["date", "district"]).dropna()
    print("shift", k, "correlation:", round(m.fc.corr(m.obs), 3))

df.to_csv("/content/merged.csv", index=False)
from google.colab import files
files.download("/content/merged.csv")

(21350, 6)
        date district    fc        obs loc_class  core
0 2019-06-01   kodagu   9.7  10.281008     ghats     0
1 2019-06-02   kodagu   3.6   0.101068     ghats     0
2 2019-06-03   kodagu  17.2   3.425805     ghats     0
3 2019-06-04   kodagu  10.9   0.000000     ghats     0
4 2019-06-05   kodagu   0.6   0.000000     ghats     0
shift -1 correlation: 0.501
shift 0 correlation: 0.651
shift 1 correlation: 0.469


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
df = pd.read_csv("/content/merged.csv", parse_dates=["date"])   # skip this line if df is still in memory
df["year"] = df.date.dt.year
TEST_YEARS = [2024, 2025]

# 1. average forecast over the 5 core districts, per day
core_all = df[df.core == 1].groupby("date").fc.mean()
core_tr  = df[(df.core == 1) & (~df.year.isin(TEST_YEARS))].groupby("date").fc.mean()

# 2. "normal" comes from training years only
mu, sd = core_tr.mean(), core_tr.std()
z = (core_all - mu) / sd              # how many SDs above or below normal

# 3. label each day
TH = 0.5
regime = pd.cut(z, [-99, -TH, TH, 99], labels=["break", "normal", "active"])
df["regime"] = df.date.map(regime).astype(str)

# 4. merge ghats and coastal into one "west" group, then make the group name
df["loc2"] = df.loc_class.replace({"ghats": "west", "coastal": "west"})
df["group"] = df.regime + "_" + df.loc2

# 5. count the heavy days per group in the training years
train = df[~df.year.isin(TEST_YEARS)]
print(train.groupby("group").apply(lambda g: pd.Series({
    "days": len(g), "heavy_64": (g.obs >= 64.5).sum(), "very_heavy_115": (g.obs >= 115.6).sum()
})))

                 days  heavy_64  very_heavy_115
group                                          
active_interior  1340        45              11
active_north      402        24               3
active_west      1608       111              30
break_interior   2170         8               1
break_north       651        46              18
break_west       2604        72              20
normal_interior  2590        25               3
normal_north      777        52              28
normal_west      3108       207              56


/tmp/ipykernel_814/3032448292.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(train.groupby("group").apply(lambda g: pd.Series({


In [10]:
# --- 1. merge thin groups ---
def final_group(g):
    if g.endswith("_north"):
        return "north"
    if g in ("break_interior", "normal_interior"):
        return "calm_interior"
    return g
df["fgroup"] = df.group.map(final_group)

train = df[~df.year.isin(TEST_YEARS)].copy()
test  = df[df.year.isin(TEST_YEARS)].copy()

# --- 2. quantile mapping ---
def fit_qm(fc, ob, n=100):
    q = np.linspace(0, 100, n)
    return np.percentile(fc, q), np.percentile(ob, q)

def apply_qm(x, table):
    return np.interp(x, table[0], table[1])

g_table = fit_qm(train.fc, train.obs)
test["global_qm"] = apply_qm(test.fc, g_table)

tables = {k: fit_qm(v.fc, v.obs) for k, v in train.groupby("fgroup")}
test["regime_qm"] = [apply_qm(x, tables.get(k, g_table)) for x, k in zip(test.fc, test.fgroup)]

# --- 3. scoring ---
def scores(fc, ob, t):
    a = ((fc >= t) & (ob >= t)).sum()      # hits
    b = ((fc >= t) & (ob <  t)).sum()      # false alarms
    c = ((fc <  t) & (ob >= t)).sum()      # misses
    n = len(fc)
    ar = (a + b) * (a + c) / n             # hits expected by chance
    safe = lambda x, y: round(x / y, 3) if y > 0 else np.nan
    return dict(POD=safe(a, a + c), FAR=safe(b, a + b), CSI=safe(a, a + b + c),
                ETS=safe(a - ar, a + b + c - ar))

rows = []
for method in ["fc", "global_qm", "regime_qm"]:
    rm = np.sqrt(((test[method] - test.obs) ** 2).mean())
    row = {"method": method, "RMSE": round(rm, 2)}
    for t in (15.6, 64.5, 115.6):
        s = scores(test[method], test.obs, t)
        row.update({f"ETS_{t}": s["ETS"], f"CSI_{t}": s["CSI"],
                    f"POD_{t}": s["POD"], f"FAR_{t}": s["FAR"]})
    rows.append(row)
pd.set_option("display.width", 250)
print(pd.DataFrame(rows).set_index("method").T)

method         fc  global_qm  regime_qm
RMSE       17.290     19.150     19.430
ETS_15.6    0.306      0.305      0.302
CSI_15.6    0.425      0.418      0.414
POD_15.6    0.618      0.579      0.574
FAR_15.6    0.423      0.400      0.402
ETS_64.5    0.218      0.250      0.237
CSI_64.5    0.230      0.267      0.254
POD_64.5    0.289      0.417      0.399
FAR_64.5    0.471      0.575      0.590
ETS_115.6   0.101      0.183      0.149
CSI_115.6   0.103      0.187      0.153
POD_115.6   0.117      0.283      0.250
FAR_115.6   0.533      0.646      0.717


In [12]:
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier

lat = districts.set_index("id").lat
LOC = {"ghats": 0, "coastal": 1, "interior": 2, "north": 3}

def feats(d, use_z=True):
    X = pd.DataFrame({
        "fc":  d.fc.values,
        "loc": d.loc_class.map(LOC).values,
        "lat": d.district.map(lat).values,
        "doy": d.date.dt.dayofyear.values,
    })
    if use_z:
        X["z"] = d.date.map(z).values        # soft regime index
    return X

def fit_gbm(tr, use_z=True):
    w = np.where(tr.obs >= 64.5, 5.0, 1.0)   # heavy days count 5x
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=4, random_state=0)
    m.fit(feats(tr, use_z), np.sqrt(tr.obs), sample_weight=w)
    return m

def predict_gbm(m, d, use_z=True):
    return np.clip(m.predict(feats(d, use_z)), 0, None) ** 2

m_z  = fit_gbm(train, True)
m_nz = fit_gbm(train, False)
test["gbm_z"]   = predict_gbm(m_z,  test, True)
test["gbm_noz"] = predict_gbm(m_nz, test, False)
test["hybrid"]  = np.where(test.global_qm >= 64.5,
                           np.maximum(test.gbm_z, test.global_qm), test.gbm_z)

rows = []
for method in ["fc", "global_qm", "regime_qm", "gbm_noz", "gbm_z", "hybrid"]:
    rm = np.sqrt(((test[method] - test.obs) ** 2).mean())
    row = {"method": method, "RMSE": round(rm, 2)}
    for t in (15.6, 64.5, 115.6):
        s = scores(test[method], test.obs, t)
        row.update({f"ETS_{t}": s["ETS"], f"POD_{t}": s["POD"], f"FAR_{t}": s["FAR"]})
    rows.append(row)
pd.set_option("display.width", 250)
print(pd.DataFrame(rows).set_index("method").T)

method         fc  global_qm  regime_qm  gbm_noz   gbm_z  hybrid
RMSE       17.290     19.150     19.430   17.930  17.790  19.350
ETS_15.6    0.306      0.305      0.302    0.329   0.332   0.332
POD_15.6    0.618      0.579      0.574    0.567   0.560   0.560
FAR_15.6    0.423      0.400      0.402    0.353   0.341   0.341
ETS_64.5    0.218      0.250      0.237    0.291   0.290   0.275
POD_64.5    0.289      0.417      0.399    0.564   0.555   0.592
FAR_64.5    0.471      0.575      0.590    0.593   0.590   0.628
ETS_115.6   0.101      0.183      0.149    0.058   0.086   0.177
POD_115.6   0.117      0.283      0.250    0.083   0.117   0.300
FAR_115.6   0.533      0.646      0.717    0.815   0.731   0.684


In [13]:
from sklearn.metrics import brier_score_loss
THR = {"heavy": 64.5, "very_heavy": 115.6}

for name, t in THR.items():
    y = (train.obs >= t).astype(int)
    m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_depth=3,
                                       l2_regularization=1.0, random_state=0)
    m.fit(feats(train, True), y)
    test[f"p_{name}"] = m.predict_proba(feats(test, True))[:, 1]

for name, t in THR.items():
    yt = (test.obs >= t).astype(int)
    p = test[f"p_{name}"]
    clim = (train.obs >= t).mean()
    bs  = brier_score_loss(yt, p)
    bs0 = brier_score_loss(yt, np.full(len(yt), clim))
    print(f"\n{name} (>= {t} mm): events in test = {int(yt.sum())}")
    print("  Brier skill score vs climatology:", round(1 - bs / bs0, 3))
    bins = pd.cut(p, [0, .02, .05, .1, .2, .4, 1.0], include_lowest=True)
    print(pd.DataFrame({"bin": bins, "pred": p, "obs": yt}).groupby("bin", observed=True).agg(
        n=("obs", "size"), mean_predicted=("pred", "mean"), actual_rate=("obs", "mean")).round(3))


heavy (>= 64.5 mm): events in test = 218
  Brier skill score vs climatology: 0.268
                   n  mean_predicted  actual_rate
bin                                              
(-0.001, 0.02]  4620           0.005        0.004
(0.02, 0.05]     601           0.032        0.033
(0.05, 0.1]      315           0.071        0.048
(0.1, 0.2]       229           0.138        0.135
(0.2, 0.4]       188           0.295        0.282
(0.4, 1.0]       147           0.549        0.537

very_heavy (>= 115.6 mm): events in test = 60
  Brier skill score vs climatology: 0.156
                   n  mean_predicted  actual_rate
bin                                              
(-0.001, 0.02]  5593           0.002        0.002
(0.02, 0.05]     194           0.031        0.005
(0.05, 0.1]      120           0.070        0.075
(0.1, 0.2]       115           0.139        0.157
(0.2, 0.4]        71           0.280        0.282
(0.4, 1.0]         7           0.468        0.286


In [14]:
import json, os, zipfile
OUT = "/content/data"
os.makedirs(OUT, exist_ok=True)

def clean(v, nd=1):
    """turn NaN into None (valid JSON) and round numbers"""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return round(float(v), nd)

def warning(mm):
    if mm >= 204.5: return "red"
    if mm >= 115.6: return "orange"
    if mm >= 64.5:  return "yellow"
    return "green"

test = test.sort_values(["date", "district"]).copy()
test["corrected"] = test["hybrid"]            # the final method shown on the site

# --- 1. districts.json ---
dist_info = df[["district", "loc_class"]].drop_duplicates().set_index("district")
dist_list = [{"id": r.id, "name": r.name, "state": r.state, "lat": r.lat, "lon": r.lon,
              "loc_class": dist_info.loc[r.id, "loc_class"]}
             for r in districts.itertuples() if r.id in dist_info.index]
json.dump(dist_list, open(f"{OUT}/districts.json", "w"))

# --- 2. forecast_by_date.json ---
by_date = {}
for d, g in test.groupby("date"):
    by_date[str(d.date())] = {
        "regime": g.regime.iloc[0],
        "districts": [{
            "id": r.district,
            "raw_mm": clean(r.fc), "corrected_mm": clean(r.corrected), "observed_mm": clean(r.obs),
            "prob": {"heavy": clean(r.p_heavy, 3), "very_heavy": clean(r.p_very_heavy, 3)},
            "warning": warning(r.corrected),
        } for r in g.itertuples()]
    }
json.dump(by_date, open(f"{OUT}/forecast_by_date.json", "w"))

# --- 3. district_series.json ---
series = {}
for dist, g in test.groupby("district"):
    series[dist] = {"dates": [str(x.date()) for x in g.date],
                    "raw": [clean(x) for x in g.fc], "corrected": [clean(x) for x in g.corrected],
                    "observed": [clean(x) for x in g.obs], "regime": list(g.regime)}
json.dump(series, open(f"{OUT}/district_series.json", "w"))

# --- 4. metrics.json ---
NAMES = {"fc": "Raw forecast", "global_qm": "Global quantile mapping",
         "regime_qm": "Regime quantile mapping", "gbm_z": "ML model (GBM)", "hybrid": "Hybrid (final)"}

def metric_block(d, method):
    out = {"rmse": clean(np.sqrt(((d[method] - d.obs) ** 2).mean()), 2)}
    for t in (15.6, 64.5, 115.6):
        s = scores(d[method], d.obs, t)
        out[f"t{t}"] = {k: (None if pd.isna(v) else float(v)) for k, v in s.items()}
        out[f"t{t}"]["events"] = int((d.obs >= t).sum())
    return out

metrics = {
  "test_period": "2024-06-01 to 2025-09-30", "train_period": "2019 to 2023",
  "n_districts": int(test.district.nunique()), "methods": NAMES,
  "overall": {m: metric_block(test, m) for m in NAMES},
  "by_regime": {r: {m: metric_block(test[test.regime == r], m) for m in NAMES}
                for r in ["active", "normal", "break"]},
  "by_location": {l: {m: metric_block(test[test.loc_class == l], m) for m in ["fc", "global_qm", "hybrid"]}
                  for l in test.loc_class.unique()},
}
prob = {}
for name, t in {"heavy": 64.5, "very_heavy": 115.6}.items():
    yt = (test.obs >= t).astype(int); p = test[f"p_{name}"]
    clim = (train.obs >= t).mean()
    bss = 1 - brier_score_loss(yt, p) / brier_score_loss(yt, np.full(len(yt), clim))
    bins = pd.cut(p, [0, .02, .05, .1, .2, .4, 1.0], include_lowest=True)
    rel = pd.DataFrame({"bin": bins, "pred": p, "obs": yt}).groupby("bin", observed=True).agg(
        n=("obs", "size"), predicted=("pred", "mean"), actual=("obs", "mean")).reset_index(drop=True)
    prob[name] = {"threshold_mm": t, "events": int(yt.sum()), "brier_skill": round(float(bss), 3),
                  "reliability": [{k: (int(v) if k == "n" else round(float(v), 3)) for k, v in row.items()}
                                  for row in rel.to_dict("records")]}
metrics["probability"] = prob
metrics["notes"] = [
  "Pilot on 25 locations: point forecasts compared with the nearest 0.25 degree IMD grid cell.",
  "Raw forecast is Open-Meteo archived model output, not NCUM.",
  "Regime label is a simplified, forecast-based index from 5 central-India locations.",
  "FSS not computed because it needs a gridded field.",
  "Test period has few very heavy events; treat those scores with caution."]
json.dump(metrics, open(f"{OUT}/metrics.json", "w"))

# --- 5. zip and download ---
with zipfile.ZipFile("/content/data.zip", "w") as zf:
    for f in os.listdir(OUT):
        zf.write(f"{OUT}/{f}", f)
for f in sorted(os.listdir(OUT)):
    print(f, round(os.path.getsize(f"{OUT}/{f}") / 1024), "KB")
from google.colab import files
files.download("/content/data.zip")

district_series.json 238 KB
districts.json 3 KB
forecast_by_date.json 864 KB
metrics.json 10 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>